# Snippet from Cookbook.md


In [ ]:
#!/usr/bin/env python3
import json
import sys
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
def load_certificate(path):
    with open(path) as f:
        cert = json.load(f)
    required = ['utility', 'drift_signal', 'constraints', 'shadow_prices']
    missing = [k for k in required if k not in cert]
    if missing:
        raise ValueError(f"Missing fields: {missing}")
    return cert
def plot_constraint_analysis(cert, output_dir='analysis'):
    Path(output_dir).mkdir(exist_ok=True)
    constraints = cert['constraints']
    names = list(constraints.keys())
    bounds = [c['bound'] for c in constraints.values()]
    actuals = [c['actual'] for c in constraints.values()]
    slacks = [c['slack'] for c in constraints.values()]
    shadows = [cert['shadow_prices'].get(n, 0) for n in names]
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    fig.suptitle('Certificate Analysis: Constraints and Signals', fontsize=16)
    x = np.arange(len(names))
    width = 0.35
    axes[0, 0].bar(x - width/2, bounds, width, label='Bound', color='orange', alpha=0.7)
    axes[0, 0].bar(x + width/2, actuals, width, label='Actual', color='green', alpha=0.7)
    axes[0, 0].set_ylabel('Value')
    axes[0, 0].set_title('Bounds vs. Actual')
    axes[0, 0].set_xticks(x)
    axes[0, 0].set_xticklabels(names, rotation=45, ha='right')
    axes[0, 0].legend()
    axes[0, 0].grid(alpha=0.3)
    colors = ['green' if s >= 0.1 else 'orange' if s >= 0 else 'red' for s in slacks]
    axes[0, 1].barh(names, slacks, color=colors, alpha=0.7)
    axes[0, 1].axvline(x=0, color='red', linestyle='--', linewidth=2)
    axes[0, 1].set_xlabel('Slack (≥0 = feasible)')
    axes[0, 1].set_title('Slack Analysis')
    axes[0, 1].grid(alpha=0.3, axis='x')
    colors_shadow = ['red' if s > 0.5 else 'orange' if s > 0.1 else 'green' for s in shadows]
    axes[1, 0].bar(names, shadows, color=colors_shadow, alpha=0.7)
    axes[1, 0].axhline(y=0.5, color='red', linestyle='--', alpha=0.5, label='High tension')
    axes[1, 0].axhline(y=0.1, color='orange', linestyle='--', alpha=0.5, label='Active')
    axes[1, 0].set_ylabel('Shadow Price λ')
    axes[1, 0].set_title('Sensitivity (Higher = More Limiting)')
    axes[1, 0].set_xticklabels(names, rotation=45, ha='right')
    axes[1, 0].legend()
    axes[1, 0].grid(alpha=0.3)
    metrics_text = f"""
Route Quality Metrics

Utility: {cert['utility']:.2f}
Free Energy: {cert['free_energy']:.2f}
Drift Signal: {cert['drift_signal']:.3f} {'Descent' if cert['drift_signal'] < 0 else 'Check'}
Entropy: {cert['entropy']:.2f} {'Healthy' if cert['entropy'] > 1.0 else 'Low'}

Utility Components:
"""
    for k, v in cert.get('utility_components', {}).items():
        metrics_text += f" {k}: {v:.2f}\n"
    metrics_text += f"""

Status:
Feasible: {'Yes' if cert['feasible'] else 'No'}
Boundary: {'Near edge' if cert.get('boundary_flag') else 'Safe'}
Model: {cert['selected_model']}
"""
    axes[1, 1].text(0.05, 0.95, metrics_text,
                     transform=axes[1, 1].transAxes,
                     fontsize=10, verticalalignment='top',
                     fontfamily='monospace',
                     bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.3))
    axes[1, 1].axis('off')
    plt.tight_layout()
    output_path = Path(output_dir) / 'certificate_analysis.png'
    plt.savefig(output_path, dpi=150, bbox_inches='tight')
    print(f"Analysis saved to {output_path}")
    return output_path
def print_recommendations(cert):
    print("\nRecommendations:\n")
    if cert['drift_signal'] > 0:
        print("Drift positive: Energy increasing. Check trust radius or constraints.")
    elif cert['drift_signal'] < -0.05:
        print("Strong descent—system stable.")
    if cert['entropy'] < 1.0:
        print("Low entropy: Increase temperature τ.")
    elif cert['entropy'] > 2.5:
        print("High entropy: Decrease τ.")
    else:
        print("Entropy balanced.")
    high_shadows = {k: v for k, v in cert['shadow_prices'].items() if v > 0.5}
    if high_shadows:
        print("High shadow prices:")
        for name, value in high_shadows.items():
            constraint = cert['constraints'][name]
            print(f" {name}: λ={value:.2f}, actual {constraint['actual']:.3f} / bound {constraint['bound']:.3f}")
            print(f" Relaxing by 20% gains ~{value * 0.2:.2f} utility.")
    if cert.get('boundary_flag'):
        print("Boundary flag: Near edge—relax constraints or accept.")
    violations = {k: v for k, v in cert['constraints'].items() if v['slack'] < 0}
    if violations:
        print("Violations:")
        for name, data in violations.items():
            print(f" {name}: Exceeded by {-data['slack']:.3f}")
    print("\n" + "="*60)
def main():
    if len(sys.argv) < 2:
        print("Usage: python analyze_certificate.py <trace.json>")
        sys.exit(1)
    cert_path = sys.argv[1]
    cert = load_certificate(cert_path)
    plot_constraint_analysis(cert)
    print_recommendations(cert)
    print(f"\nSummary:")
    print(f"Model: {cert['selected_model']}")
    print(f"Utility: {cert['utility']:.2f}")
    print(f"Feasible: {'Yes' if cert['feasible'] else 'No'}")
if __name__ == "__main__":
    main()
